# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiwateNandini/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client on one reporting date (client_hash_id × content_hash_id × report_date) in fact_content_daily_performance.

Time window: For this data-contract iteration, I will use the March 2026 partition (month=2026-03). I will use this mid-panel month to verify the data structure and feature availability rather than using the final June 2026 month, which should remain a sealed outcome period.

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("March 2026 partition connected.")

March 2026 partition connected.


In [4]:
from google.colab import userdata
import duckdb

# Get HF token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Go to Colab → Secrets → add a secret named HF_TOKEN."
    )

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("✅ HF token loaded")
print("✅ DuckDB connection created")
print("✅ March 2026 table configured")

✅ HF token loaded
✅ DuckDB connection created
✅ March 2026 table configured


In [5]:
test = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MARCH}
""").df()

display(test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [6]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {MARCH}
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain combinations:")
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations:


,client_hash_id,content_hash_id,report_date,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: Historical performance variables that are available before the decision moment, such as previous impressions, clicks, CTR, and average position.

Label / proxy: The later observed content-performance outcome that I want to study. Label-derived fields such as trend_direction and trend_pct will not be used as features.

Context: client_hash_id, content_hash_id, and report_date. These are useful for grouping, joining, filtering, and validation, but they are not model features.

Excluded: Future-period performance and any field derived from the outcome. These are excluded because they would not be known at the decision moment and could cause leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#Query 1 — Grain

In [7]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM {MARCH}
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Rows violating the expected grain:")
display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the expected grain:


,client_hash_id,content_hash_id,report_date,row_count


#Query 2 — Count and date window

In [8]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {MARCH}
""").df()

display(window_check)

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


#Query 3 — Missing values

In [10]:
columns = con.sql(f"""
DESCRIBE SELECT *
FROM {MARCH}
""").df()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
missingness = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    AVG(CASE
        WHEN gsc_impressions IS NULL THEN 1.0
        ELSE 0.0
    END) AS impressions_missing_pct,

    AVG(CASE
        WHEN gsc_clicks IS NULL THEN 1.0
        ELSE 0.0
    END) AS clicks_missing_pct,

    AVG(CASE
        WHEN gsc_avg_position IS NULL THEN 1.0
        ELSE 0.0
    END) AS position_missing_pct

FROM {MARCH}
""").df()

display(missingness)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,impressions_missing_pct,clicks_missing_pct,position_missing_pct
0,9841378,0.0,0.0,0.633074


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has several limitations. History depth differs across clients, so the same calendar window does not necessarily represent the same amount of historical information for every client. Some rows also have limited analytics availability, so GA4-related values cannot automatically be interpreted as meaningful zeros. The query-level table has a fixed 90-day window that can overlap with outcome periods, so its fields require careful time alignment before being used as features.

The analysis can provide observed and directional decision-support signals, but it cannot establish that changing a page will cause a particular Google ranking or traffic outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.